In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime
from pathlib import Path
import os

UPSTREAM_COMMIT = "b76b93da8bd3096a99964a96ae29d46f197a0651"
DATA_ROOT = Path("/content/drive/MyDrive/datasets/mvtec_anomaly_detection")
DATA_SOURCE = "official_mvtec_manual_download"
PROJECT_ROOT = Path("/content/drive/MyDrive/CS2203_MuSc")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_E00_mvtec_bottle_clip_l14_336_baseline")
RUN_ROOT = PROJECT_ROOT / "runs" / RUN_ID
OUTPUT_ROOT = RUN_ROOT / "output"

RUN_ROOT.mkdir(parents=True, exist_ok=False)
os.environ["MUSC_COMMIT"] = UPSTREAM_COMMIT
os.environ["MUSC_DATA_ROOT"] = str(DATA_ROOT)
os.environ["MUSC_RUN_ROOT"] = str(RUN_ROOT)
os.environ["MUSC_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

print("Run ID:", RUN_ID)
print("Dataset:", DATA_ROOT)
print("Persistent output:", RUN_ROOT)

Mounted at /content/drive
Run ID: 20261011_161100_E00_mvtec_bottle_clip_l14_336_baseline
Dataset: /content/drive/MyDrive/datasets/mvtec_anomaly_detection
Persistent output: /content/drive/MyDrive/CS2203_MuSc/runs/20261011_161100_E00_mvtec_bottle_clip_l14_336_baseline


In [ ]:
import platform
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "GPU chưa được bật. Hãy đổi Colab runtime sang GPU."

gpu_name = torch.cuda.get_device_name(0)
environment_text = "\n".join([
    f"python={sys.version}",
    f"platform={platform.platform()}",
    f"torch={torch.__version__}",
    f"cuda_runtime={torch.version.cuda}",
    f"gpu={gpu_name}",
    f"upstream_commit={UPSTREAM_COMMIT}",
])

print(environment_text)
(RUN_ROOT / "environment-before-install.txt").write_text(environment_text, encoding="utf-8")
subprocess.run(["nvidia-smi"], check=True)

python=3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
platform=Linux-6.6.122+-x86_64-with-glibc2.39
torch=2.11.0+cu130
cuda_runtime=13.0
gpu=Tesla T4
upstream_commit=b76b93da8bd3096a99964a96ae29d46f197a0651


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
## Tải MVTec AD từ Hugging Face
import os
import subprocess
import sys
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "huggingface_hub"],
    check=True,
)

from huggingface_hub import HfApi, hf_hub_download

HF_DATASET_REPO = "micguida1/mvtech_anomaly_detection"
HF_DATASET_REVISION = HfApi().dataset_info(HF_DATASET_REPO).sha
HF_CACHE = PROJECT_ROOT / "cache" / "huggingface"

archive_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    filename="mvtec_anomaly_detection.tar.xz",
    revision=HF_DATASET_REVISION,
    cache_dir=HF_CACHE,
)

extract_root = Path("/content/data")
extract_root.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ["tar", "-xJf", archive_path, "-C", str(extract_root)],
    check=True,
)

expected_root = extract_root / "mvtec_anomaly_detection"
if expected_root.is_dir():
    DATA_ROOT = expected_root
else:
    candidates = [path.parent.parent for path in extract_root.rglob("bottle/test")]
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, f"Không xác định được duy nhất dataset root: {candidates}"
    DATA_ROOT = candidates[0]

DATA_SOURCE = f"huggingface:{HF_DATASET_REPO}@{HF_DATASET_REVISION}"
os.environ["MUSC_DATA_ROOT"] = str(DATA_ROOT)

(RUN_ROOT / "dataset-source.txt").write_text(
    f"source={DATA_SOURCE}\narchive=mvtec_anomaly_detection.tar.xz\n",
    encoding="utf-8",
)

print("Dataset source:", DATA_SOURCE)
print("Extracted dataset:", DATA_ROOT)

mvtec_anomaly_detection.tar.xz: reconstructing file:   0%|          |  0.00B / 5.26GB            

mvtec_anomaly_detection.tar.xz: downloading bytes:           |  0.00B            

Dataset source: huggingface:micguida1/mvtech_anomaly_detection@eb692b36147a3b3f12c561b879a062104c1f20b8
Extracted dataset: /content/data


In [ ]:
## Kiểm tra cấu trúc và số lượng ảnh `bottle`
required_paths = [
    DATA_ROOT / "bottle" / "ground_truth",
    DATA_ROOT / "bottle" / "test",
    DATA_ROOT / "bottle" / "train",
]

for path in required_paths:
    assert path.is_dir(), f"Thiếu thư mục bắt buộc: {path}"

test_images = list((DATA_ROOT / "bottle" / "test").glob("*/*"))
assert test_images, "Không tìm thấy ảnh test của bottle."
assert len(test_images) == 83, f"Cần 83 ảnh test bottle, hiện có {len(test_images)}."
print(f"Đã tìm thấy {len(test_images)} ảnh test bottle.")

Đã tìm thấy 83 ảnh test bottle.


In [ ]:
# Clone MuSc và khóa đúng commit
%%bash
set -euo pipefail
rm -rf /content/MuSc
git clone https://github.com/xrli-U/MuSc.git /content/MuSc
cd /content/MuSc
git checkout --detach "$MUSC_COMMIT"
git rev-parse HEAD | tee "$MUSC_RUN_ROOT/upstream-commit.txt"

b76b93da8bd3096a99964a96ae29d46f197a0651


Cloning into '/content/MuSc'...
HEAD is now at b76b93d Add files via upload


In [ ]:
# Cài các thư viện còn thiếu
import subprocess
import sys

packages = [
    "ftfy==6.1.3",
    "openpyxl==3.1.2",
    "timm==0.9.12",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'ftfy==6.1.3', 'openpyxl==3.1.2', 'timm==0.9.12'], returncode=0)

In [ ]:
# Lưu các phiên bản thư viện
import cv2
import ftfy
import openpyxl
import platform
import sklearn
import skimage
import sys
import timm
import torch
import torchvision

versions = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "opencv": cv2.__version__,
    "ftfy": ftfy.__version__,
    "openpyxl": openpyxl.__version__,
    "scikit_learn": sklearn.__version__,
    "scikit_image": skimage.__version__,
    "timm": timm.__version__,
    "upstream_commit": UPSTREAM_COMMIT,
    "dataset_source": DATA_SOURCE,
}

environment_text = "\n".join(f"{key}={value}" for key, value in versions.items())
print(environment_text)
(RUN_ROOT / "environment.txt").write_text(environment_text, encoding="utf-8")

python=3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
platform=Linux-6.6.122+-x86_64-with-glibc2.39
torch=2.11.0+cu130
torchvision=0.26.0+cu130
cuda_runtime=13.0
gpu=Tesla T4
opencv=4.14.0
ftfy=6.1.2
openpyxl=3.1.2
scikit_learn=1.6.1
scikit_image=0.25.2
timm=0.9.12
upstream_commit=b76b93da8bd3096a99964a96ae29d46f197a0651
dataset_source=huggingface:micguida1/mvtech_anomaly_detection@eb692b36147a3b3f12c561b879a062104c1f20b8


429

In [ ]:
# Tạo và lưu lệnh chạy baseline
command = f"""python examples/musc_main.py \\
  --device 0 \\
  --data_path {DATA_ROOT} \\
  --dataset_name mvtec_ad \\
  --class_name bottle \\
  --backbone_name ViT-L-14-336 \\
  --pretrained openai \\
  --feature_layers 5 11 17 23 \\
  --img_resize 518 \\
  --divide_num 1 \\
  --r_list 1 3 5 \\
  --batch_size 4 \\
  --output_dir {OUTPUT_ROOT} \\
  --vis True \\
  --vis_type whole_norm \\
  --save_excel True"""

print(command)
(RUN_ROOT / "command.sh").write_text(
    "#!/usr/bin/env bash\nset -euo pipefail\n" + command + "\n",
    encoding="utf-8",
)

python examples/musc_main.py \
  --device 0 \
  --data_path /content/data \
  --dataset_name mvtec_ad \
  --class_name bottle \
  --backbone_name ViT-L-14-336 \
  --pretrained openai \
  --feature_layers 5 11 17 23 \
  --img_resize 518 \
  --divide_num 1 \
  --r_list 1 3 5 \
  --batch_size 4 \
  --output_dir /content/drive/MyDrive/CS2203_MuSc/runs/20261011_161100_E00_mvtec_bottle_clip_l14_336_baseline/output \
  --vis True \
  --vis_type whole_norm \
  --save_excel True


513

In [ ]:
## Sửa xung đột tên package `datasets` trên Colab
from pathlib import Path
import subprocess
import sys

repo_root = Path("/content/MuSc")
entrypoint = repo_root / "examples" / "musc_main.py"
local_package = repo_root / "datasets" / "__init__.py"

source = entrypoint.read_text(encoding="utf-8")
old_line = "sys.path.append(os.getcwd())"
new_line = "sys.path.insert(0, os.getcwd())"

if old_line in source:
    entrypoint.write_text(source.replace(old_line, new_line, 1), encoding="utf-8")
elif new_line not in source:
    raise RuntimeError("Không tìm thấy dòng sys.path mong đợi trong musc_main.py")

local_package.touch(exist_ok=True)

diagnostic = subprocess.run(
    [
        sys.executable,
        "-c",
        "import datasets; print(datasets.__file__); "
        "import datasets.mvtec; print('Đã import datasets.mvtec thành công')",
    ],
    cwd=repo_root,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
)
print(diagnostic.stdout)

resolved_package = Path(diagnostic.stdout.splitlines()[0]).resolve()
assert repo_root.resolve() in resolved_package.parents, (
    f"Python vẫn đang dùng package datasets bên ngoài MuSc: {resolved_package}"
)

git_status = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo_root,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout
git_diff = subprocess.run(
    ["git", "diff", "--", "examples/musc_main.py"],
    cwd=repo_root,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout

(RUN_ROOT / "compatibility-fix.txt").write_text(
    "Reason: avoid collision with the Hugging Face datasets package.\n"
    "Effect on method: none; import resolution only.\n\n"
    + git_status
    + "\n"
    + git_diff,
    encoding="utf-8",
)

/content/MuSc/datasets/__init__.py
Đã import datasets.mvtec thành công



8869

In [ ]:
# Chạy E00 và ghi log
import subprocess
import time

log_path = RUN_ROOT / "console.log"
started = time.time()

with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        ["bash", str(RUN_ROOT / "command.sh")],
        cwd="/content/MuSc",
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

    return_code = process.wait()

elapsed = time.time() - started
(RUN_ROOT / "elapsed-seconds.txt").write_text(f"{elapsed:.3f}\n", encoding="utf-8")
print(f"Thời gian chạy: {elapsed / 60:.2f} phút")

if return_code != 0:
    raise RuntimeError(
        f"MuSc kết thúc với mã lỗi {return_code}. "
        f"Hãy kiểm tra {log_path} trước khi đổi cấu hình."
    )

xFormers not available
xFormers not available
{'datasets': {'dataset_name': 'mvtec_ad', 'data_path': '/content/data', 'class_name': 'bottle', 'img_resize': 518, 'divide_num': 1}, 'models': {'backbone_name': 'ViT-L-14-336', 'pretrained': 'openai', 'batch_size': 4, 'feature_layers': [5, 11, 17, 23], 'r_list': [1, 3, 5]}, 'device': '0', 'testing': {'output_dir': '/content/drive/MyDrive/CS2203_MuSc/runs/20261011_161100_E00_mvtec_bottle_clip_l14_336_baseline/output', 'vis': True, 'vis_type': 'whole_norm', 'save_excel': True}}

  0%|                                               | 0.00/934M [00:00<?, ?iB/s]
  0%|                                      | 2.52M/934M [00:00<00:36, 25.2MiB/s]
  1%|▍                                     | 11.1M/934M [00:00<00:15, 60.9MiB/s]
  2%|▋                                     | 17.2M/934M [00:01<01:57, 7.78MiB/s]
  2%|▊                                     | 20.7M/934M [00:02<02:00, 7.60MiB/s]
  3%|█▏                                    | 27.7M/934M [00:02<01:1

In [ ]:
# Đọc file kết quả
from IPython.display import display
import pandas as pd

result_file = OUTPUT_ROOT / "mvtec_ad" / "ViT-L-14-336" / "imagesize518" / "results.xlsx"
assert result_file.is_file(), f"Không tìm thấy file kết quả: {result_file}"

results = pd.read_excel(result_file)
display(results)

image_dir = OUTPUT_ROOT / "mvtec_ad" / "ViT-L-14-336" / "imagesize518" / "bottle"
image_files = sorted(path for path in image_dir.rglob("*") if path.is_file())

print("Số anomaly map đã lưu:", len(image_files))
print("Thư mục của lần chạy:", RUN_ROOT)

,Unnamed: 0,auroc_px,f1_px,ap_px,aupro,auroc_sp,f1_sp,ap_sp
0,bottle,98.477734,79.173999,83.04498,96.10451,99.920635,99.212598,99.975198
1,mean,98.477734,79.173999,83.04498,96.10451,99.920635,99.212598,99.975198


Số anomaly map đã lưu: 83
Thư mục của lần chạy: /content/drive/MyDrive/CS2203_MuSc/runs/20261011_161100_E00_mvtec_bottle_clip_l14_336_baseline


In [ ]:
# so sánh với kết quả tham chiếu
import pandas as pd

reference = pd.DataFrame([{
    "category": "bottle",
    "auroc_px": 98.48,
    "f1_px": 79.17,
    "ap_px": 83.04,
    "aupro": 96.10,
    "auroc_sp": 99.92,
    "f1_sp": 99.21,
    "ap_sp": 99.98,
}])

reproduced = pd.read_excel(result_file).iloc[[0]].copy()
reproduced = reproduced.rename(columns={reproduced.columns[0]: "category"})
reproduced.insert(0, "source", "reproduced_E00")
reference.insert(0, "source", "official_repository")

comparison = pd.concat([reference, reproduced], ignore_index=True)
display(comparison)
comparison.to_csv(RUN_ROOT / "reference-comparison.csv", index=False)

,source,category,auroc_px,f1_px,ap_px,aupro,auroc_sp,f1_sp,ap_sp
0,official_repository,bottle,98.480000,79.170000,83.04000,96.10000,99.920000,99.210000,99.980000
1,reproduced_E00,bottle,98.477734,79.173999,83.04498,96.10451,99.920635,99.212598,99.975198
